In [2]:
from google.colab import drive
import zipfile
import os

# Connect to Google Drive
drive.mount('/content/drive')

# Path matches the folder shortcut and the exact zip filename
zip_path = '/content/drive/MyDrive/food 101 sub-dataset/food101_subset.zip'
extract_path = '/content/dataset'

# Extract the images
print("Extracting images... this might take a minute.")
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)
print("Done! Folders inside:", os.listdir(extract_path))

Mounted at /content/drive
Extracting images... this might take a minute.
Done! Folders inside: ['val', 'train', 'test']


In [8]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
from torch.utils.data import DataLoader
import os

# 1. Check for the GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on: {device}")

# 2. Prepare the Images (EfficientNet needs them resized to 224x224)
transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Define paths matching your exact folder structure
dataset_dir = '/content/dataset'
train_dir = os.path.join(dataset_dir, 'train')
val_dir = os.path.join(dataset_dir, 'val') # Updated to match your folder exactly

# Load the datasets
train_data = datasets.ImageFolder(train_dir, transform=transform)
val_data = datasets.ImageFolder(val_dir, transform=transform)

train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
val_loader = DataLoader(val_data, batch_size=32, shuffle=False)

num_classes = len(train_data.classes)
print(f"Found {num_classes} food categories to train on.")

# 3. Setup EfficientNet-B0
model = efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)

# Freeze the base layers to speed up training
for param in model.parameters():
    param.requires_grad = False

# Update the final classification layer for our specific food classes
model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
model = model.to(device)

# 4. Set up the optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.classifier.parameters(), lr=0.001)

# 5. Start the Training Loop (3 Epochs)
epochs = 3
for epoch in range(epochs):
    model.train()
    running_loss = 0.0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch {epoch+1}/{epochs} completed. Training Loss: {running_loss/len(train_loader):.4f}")

# Save the finished model
torch.save(model.state_dict(), '/content/efficientnet_food101.pth')
print("Model saved as 'efficientnet_food101.pth'!")

Training on: cuda
Found 20 food categories to train on.
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 105MB/s]


Epoch 1/3 completed. Training Loss: 1.6926
Epoch 2/3 completed. Training Loss: 1.1531
Epoch 3/3 completed. Training Loss: 1.0347
Model saved as 'efficientnet_food101.pth'!


In [9]:
# Set the model to evaluation mode
model.eval()

correct = 0
total = 0

print("Testing against validation data...")

# Turn off gradient calculations for faster evaluation
with torch.no_grad():
    for inputs, labels in val_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        # Get the model's predictions
        outputs = model(inputs)
        _, predicted = torch.max(outputs.data, 1)

        # Tally up correct predictions
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f"Final Validation Accuracy: {accuracy:.2f}%")

Testing against validation data...
Final Validation Accuracy: 71.64%
